# Extract Focal Plane Geometry

- author Sylvie Dagoret-Campagne
- creation date 2026-04-02
- LSST pipelines : w_2026_10


Extract CCD geometry (center positions) from LSST camera.
Run this at SLAC and copy the output file back.
- https://github.com/PFLeget/dp2_psf/blob/master/rayTracingFocalPlane/extract_ccd_geometry.py

## Import

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees


from lsst.skymap import PatchInfo, Index2D

from lsst.afw.math import binImage

import lsst.afw.cameraGeom as cameraGeom
import lsst.geom as geom

In [ ]:
import gc

In [ ]:
from astropy.visualization import ZScaleInterval

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [ ]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Config

In [ ]:
# The output repo is tagged with the Jira ticket number "DM-40356":
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# bad : crash collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'

# bad : collection = "LSSTComCam/runs/DRP/20241101_20241211/w_2024_51/DM-48233"

# not working perhaps because I am using w_2025_10 version
# bad : no ccd visit collection = "LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864"
# bad : no ccd visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_15/DM-50050'
# bad : no cce visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864'
# bad : no cce visit collection collection = 'LSSTComCam/runs/DRP/DP1/w_2025_13/DM-49751'
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + instrument + "'"
collectionStr = collection[-1].replace("/", "_")
BANDSEL = "r"  # Most fields were observed in red filter

In [ ]:
# Initialize the butler repo:
butler = Butler(repo, collections=collection)
registry = butler.registry

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

In [ ]:
camera = butler.get("camera", collections=collection, instrument=instrument)

In [ ]:
# print(camera.getName(),camera.getNameMap())

## start Focal Plane Extraction

In [ ]:
data = []
for det in camera:
    det_id = det.getId()
    name = det.getName()
    # Get detector center in focal plane coordinates (mm)
    # Using the center pixel transformed to FOCAL_PLANE
    bbox = det.getBBox()
    center_pix_x = (bbox.getMinX() + bbox.getMaxX()) / 2
    center_pix_y = (bbox.getMinY() + bbox.getMaxY()) / 2

    transform = det.getTransform(cameraGeom.PIXELS, cameraGeom.FOCAL_PLANE)
    center_fp = transform.applyForward(geom.Point2D(center_pix_x, center_pix_y))

    # Get corners to determine orientation/size
    corners_pix = [
        geom.Point2D(bbox.getMinX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMaxY()),
        geom.Point2D(bbox.getMinX(), bbox.getMaxY()),
    ]
    corners_fp = [transform.applyForward(c) for c in corners_pix]

    data.append(
        {
            "detector": det_id,
            "name": name,
            "x_center": center_fp.getX(),
            "y_center": center_fp.getY(),
            "corner0_x": corners_fp[0].getX(),
            "corner0_y": corners_fp[0].getY(),
            "corner1_x": corners_fp[1].getX(),
            "corner1_y": corners_fp[1].getY(),
            "corner2_x": corners_fp[2].getX(),
            "corner2_y": corners_fp[2].getY(),
            "corner3_x": corners_fp[3].getX(),
            "corner3_y": corners_fp[3].getY(),
        }
    )

## Save file

In [ ]:
# Save to CSV
import csv

with open("ccd_geometry.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(data)

print(f"Saved {len(data)} detectors to ccd_geometry.csv")
print("Copy this file back to rayTracingFocalPlane/data/")

In [ ]:
len(data)

## Plot Focal Plane

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def plot_ccd_geometry(
    geometry_csv,
    show_labels=True,
    label_detector=True,
    label_name=True,
    figsize=(10, 10),
    fontsize=6,
    invert_x=False,
    invert_y=False,
    title="CCD Geometry (LSST Focal Plane)",
):
    """
    Affiche la géométrie des CCD avec labels pour validation.

    Parameters
    ----------
    geometry_csv : str
        Fichier CSV contenant les corners
    show_labels : bool
        Affiche les labels au centre des CCD
    label_detector : bool
        Affiche le numéro de détecteur
    label_name : bool
        Affiche le nom (Rxx_Syy)
    """

    # --- Load geometry
    geom = pd.read_csv(geometry_csv)

    fig, ax = plt.subplots(figsize=figsize)

    for _, row in geom.iterrows():
        # --- corners
        corners = [
            (row["corner0_x"], row["corner0_y"]),
            (row["corner1_x"], row["corner1_y"]),
            (row["corner2_x"], row["corner2_y"]),
            (row["corner3_x"], row["corner3_y"]),
        ]

        # --- draw CCD polygon
        poly = patches.Polygon(corners, facecolor="none", edgecolor="black", linewidth=0.5)
        ax.add_patch(poly)

        # --- draw center
        ax.plot(row["x_center"], row["y_center"], marker="o", markersize=2)

        # --- labels
        if show_labels:
            label_parts = []
            if label_detector:
                label_parts.append(f"{int(row['detector'])}")
            if label_name:
                label_parts.append(row["name"])

            label = "\n".join(label_parts)

            ax.text(row["x_center"], row["y_center"], label, ha="center", va="center", fontsize=fontsize)

    # --- aesthetics
    ax.set_aspect("equal")
    ax.set_title(title)

    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")

    # Optional axis inversion (debug orientation)
    if invert_x:
        ax.invert_xaxis()
    if invert_y:
        ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

In [ ]:
# plot_ccd_geometry("ccd_geometry.csv", invert_x=True)
plot_ccd_geometry("ccd_geometry.csv")

## Plot Heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def plot_focal_plane_heatmap(
    alerts_df,
    geometry_csv,
    value_col=None,  # None = counts, sinon colonne à agréger
    agg_func="count",  # "count", "mean", etc.
    cmap="viridis",
    log_scale=False,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
):
    """
    Affiche une heatmap de la surface focale LSST.

    Parameters
    ----------
    alerts_df : pd.DataFrame
        Doit contenir une colonne 'detector'
    geometry_csv : str
        Chemin vers ton fichier ccd_geometry.csv
    value_col : str or None
        Si None → compte le nombre d’objets par CCD
        Sinon → agrège cette colonne
    agg_func : str
        Fonction d’agrégation ('count', 'mean', etc.)
    """

    # --- Load geometry
    geom = pd.read_csv(geometry_csv)

    # --- Compute values per detector
    if value_col is None:
        values = alerts_df.groupby("detector").size()
    else:
        values = alerts_df.groupby("detector")[value_col].agg(agg_func)

    # --- Merge with geometry
    geom["value"] = geom["detector"].map(values).fillna(0)

    # --- Optional log scale
    if log_scale:
        geom["value"] = np.log10(geom["value"] + 1)

    # --- Normalize colormap
    vmin = geom["value"].min()
    vmax = geom["value"].max()

    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap)

    # --- Plot
    fig, ax = plt.subplots(figsize=figsize)

    for _, row in geom.iterrows():
        corners = [
            (row["corner0_x"], row["corner0_y"]),
            (row["corner1_x"], row["corner1_y"]),
            (row["corner2_x"], row["corner2_y"]),
            (row["corner3_x"], row["corner3_y"]),
        ]

        color = cmap(norm(row["value"]))

        poly = patches.Polygon(corners, facecolor=color, edgecolor="black", linewidth=0.2)
        ax.add_patch(poly)

    # --- Aesthetic
    ax.set_aspect("equal")
    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")
    ax.set_title(title)

    # Remove axes ticks (optional)
    ax.set_xticks([])
    ax.set_yticks([])

    # --- Colorbar
    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
# plot_focal_plane_heatmap(
#    alerts_df,
#    "ccd_geometry.csv",
#    title="Nombre d'alertes par CCD"
# )

In [ ]:
# plot_focal_plane_heatmap(
#    alerts_df,
#    "ccd_geometry.csv",
#    value_col="snr",
#    agg_func="mean",
#    title="SNR moyen par CCD"
# )

In [ ]:
# plot_focal_plane_heatmap(
#    alerts_df,
#    "ccd_geometry.csv",
#    log_scale=True,
#    title="Alertes (log10)"
# )

In [ ]:
# ax.text(row["x_center"], row["y_center"], row["name"],
#        ha="center", va="center", fontsize=4)

In [ ]:
# aggregation par raft
# alerts_df["raft"] = alerts_df["detector_name"].str.split("_").str[0]
# alerts_df.groupby("raft").size()

In [ ]:
# detection de probleme
# threshold = np.percentile(geom["value"], 5)
# bad_ccd = geom[geom["value"] < threshold]
# print(bad_ccd[["detector", "name", "value"]])